# Batch 06 — Exp01: `v3_behavioral_proxy` (Cognitive Load Behavioral Proxy)

Notebook mandiri untuk eksperimen klasifikasi 3-kelas (`Low`, `Medium`, `High`) pada paket **`v3_behavioral_proxy`** (`window_15s_step_5s/good_acceptable.csv`).

**Kontrak Anti-Kebocoran:**
1. Pembagian peserta (`Train` 16 orang, `validation` 3 orang, `Test` 3 orang) dari `participant_splits.csv` dipertahankan beku.
2. Inner-CV di dalam Train (5-fold `GroupKFold` per `participant_id`) **menghitung ulang referensi waktu dan label `behavioral_concordance_v2` dari `episode_labels.csv` menggunakan inner-Train saja**, beserta bobot sampel inner-Train.
3. Hanya fitur sensor murni yang masuk ke `X` (`core10`, `core9` tanpa fitur iris redundan, dan `core9_head4`).
4. Bobot sampel hanya digunakan saat fitting Train—tidak pernah pada validation atau Test.
5. Satu pemenang dikunci dari **Validation** (dengan kontrol OOF inner-CV) sebelum mengevaluasi **Test** tepat satu kali.


In [1]:
# Cell 1: Konfigurasi Lingkungan, Seed, dan Verifikasi Integritas Dataset
import hashlib
import json
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import platform
import random
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
if "ipykernel" not in os.sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    recall_score,
)
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
LABELS = ["Low", "Medium", "High"]
LABEL_TO_IDX = {lbl: i for i, lbl in enumerate(LABELS)}

def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)
torch.set_num_threads(2)

# Auto-detect lokal vs Google Colab
IN_COLAB = "google.colab" in os.sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Penelitian-Adaptive-IDE")
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"
else:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        Path(r"e:\Skripsi\ngodong\Adaptive-IDE"),
    ]
    PROJECT_ROOT = next((p for p in candidates if (p / "models/batch-06/dataset").exists()), cwd)
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"

V3_DIR = DATASET_ROOT / "v3_behavioral_proxy"
OUTPUT_DIR = PROJECT_ROOT / "models/batch-06/outputs/exp01_v3_behavioral_proxy"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "window_15s_step_5s/good_acceptable.csv",
    "participant_splits.csv",
    "episode_labels.csv",
    "window_15s_step_5s/window_assignments.csv",
    "window_15s_step_5s/train_sample_weights.csv",
]

file_hashes = {}
for rel in REQUIRED_FILES:
    p = V3_DIR / rel
    assert p.exists(), f"File wajib tidak ditemukan: {p}"
    file_hashes[rel] = hashlib.sha256(p.read_bytes()).hexdigest()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"V3_DIR       : {V3_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"DEVICE       : {DEVICE} | Python {platform.python_version()} | PyTorch {torch.__version__}")
print("SHA-256 dataset v3 terverifikasi (5/5 file).")


PROJECT_ROOT : E:\Skripsi\ngodong\Adaptive-IDE
V3_DIR       : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\dataset\v3_behavioral_proxy
OUTPUT_DIR   : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp01_v3_behavioral_proxy
DEVICE       : cpu | Python 3.10.11 | PyTorch 2.11.0+cpu
SHA-256 dataset v3 terverifikasi (5/5 file).


In [2]:
# Cell 2: Muat & Gabungkan CSV V3 lewat Kunci + Audit Integritas & Bobot
CORE10 = [
    "rmssd_ms",
    "sdnn_ms",
    "blink_rate_per_minute",
    "ear_left_mean",
    "ear_left_std",
    "ear_right_mean",
    "ear_right_std",
    "iris_size_estimate_mean",
    "iris_size_estimate_std",
    "iris_size_delta_mean",
]
CORE9 = [c for c in CORE10 if c != "iris_size_delta_mean"]
HEAD4 = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
FEATURE_PROFILES = {
    "core10": CORE10,
    "core9": CORE9,
    "core9_head4": CORE9 + HEAD4,
}

raw_df = pd.read_csv(V3_DIR / "window_15s_step_5s/good_acceptable.csv")
splits_df = pd.read_csv(V3_DIR / "participant_splits.csv")
episodes_df = pd.read_csv(V3_DIR / "episode_labels.csv")
wa_df = pd.read_csv(V3_DIR / "window_15s_step_5s/window_assignments.csv")
weights_df = pd.read_csv(V3_DIR / "window_15s_step_5s/train_sample_weights.csv")

assert raw_df["window_id"].is_unique, "window_id duplikat!"
df = raw_df.merge(
    wa_df[["window_id", "episode_id", "split"]],
    on="window_id",
    how="left",
    validate="1:1",
)
df = df.merge(
    splits_df.rename(columns={"split": "participant_split"}),
    on="participant_id",
    how="left",
    validate="m:1",
)
assert df["episode_id"].notna().all(), "Ada window tanpa episode_id!"
assert (df["split"] == df["participant_split"]).all(), "Ketidakcocokan split peserta vs window_assignments!"
assert df.groupby("participant_id")["split"].nunique().max() == 1, "Kebocoran peserta antar-split!"
assert df.groupby("episode_id")["split"].nunique().max() == 1, "Kebocoran episode antar-split!"

for feat_name, cols in FEATURE_PROFILES.items():
    arr = df[cols].to_numpy(dtype=float)
    assert np.isfinite(arr).all(), f"Nilai non-finite pada {feat_name}!"

df = df.merge(
    weights_df[["window_id", "sample_weight"]],
    on="window_id",
    how="left",
    validate="1:1",
)
is_train = df["split"] == "Train"
assert df.loc[is_train, "sample_weight"].notna().all(), "Baris Train tanpa sample_weight!"
assert df.loc[~is_train, "sample_weight"].isna().all(), "Validation/Test tidak boleh punya Train sample_weight!"

# Urutkan kronologis per sesi untuk normalisasi kausal
df["window_start_dt"] = pd.to_datetime(df["window_start"])
df = df.sort_values(["session_id", "window_start_dt", "window_id"]).reset_index(drop=True)

print(f"Total rows: {len(df)} | Participants: {df['participant_id'].nunique()} | Episodes: {df['episode_id'].nunique()}")
for sp in ["Train", "validation", "Test"]:
    sub = df[df["split"] == sp]
    cls_str = ", ".join(
        f"{c}={len(sub[sub['cognitive_load_label']==c])}w/{sub[sub['cognitive_load_label']==c]['episode_id'].nunique()}ep/{sub[sub['cognitive_load_label']==c]['participant_id'].nunique()}p"
        for c in LABELS
    )
    print(f"  {sp:10s}: {len(sub):4d}w, {sub['episode_id'].nunique():2d}ep, {sub['participant_id'].nunique():2d}p | {cls_str}")

w_tr = df.loc[df["split"] == "Train", "sample_weight"].to_numpy()
print(f"Train sample_weight stats: min={w_tr.min():.4f}, median={np.median(w_tr):.4f}, max={w_tr.max():.4f} (ratio={w_tr.max()/w_tr.min():.1f}x)")


Total rows: 1193 | Participants: 21 | Episodes: 66
  Train     :  744w, 37ep, 15p | Low=54w/12ep/7p, Medium=134w/9ep/6p, High=556w/16ep/10p
  validation:  218w, 11ep,  3p | Low=14w/4ep/3p, Medium=53w/3ep/2p, High=151w/4ep/2p
  Test      :  231w, 18ep,  3p | Low=62w/12ep/3p, Medium=50w/4ep/2p, High=119w/2ep/2p
Train sample_weight stats: min=0.1653, median=0.4960, max=35.4286 (ratio=214.3x)


In [3]:
# Cell 3: Rekomputasi Label & Bobot Inner-CV Bebas Kebocoran + Normalisasi & Bobot
MIN_REFERENCE = 5

def relabel_episodes_inner(ep_table: pd.DataFrame, inner_train_participants: set) -> dict:
    """Hitung ulang referensi waktu dan label behavioral_concordance_v2 HANYA dari inner-Train."""
    ref_items = defaultdict(list)
    for row in ep_table.itertuples(index=False):
        if (
            row.participant_id in inner_train_participants
            and bool(row.valid)
            and bool(row.completed)
            and bool(row.reference_eligible)
            and row.success_submission_type == "manual"
            and float(row.active_seconds) > 0.0
        ):
            ref_items[row.item_key].append((row.participant_id, float(row.active_seconds)))

    labels_out = {}
    for row in ep_table.itertuples(index=False):
        if not bool(row.valid):
            continue
        peers = [sec for pid, sec in ref_items.get(row.item_key, []) if pid != row.participant_id]
        if len(peers) < MIN_REFERENCE:
            continue
        lower, upper = np.quantile(peers, [1.0 / 3.0, 2.0 / 3.0], method="linear")
        if not (0.0 < lower < upper):
            continue
        sec = float(row.active_seconds)
        errs = int(row.manual_errors)
        t_band = "Low" if sec <= lower else ("Medium" if sec <= upper else "High")
        if bool(row.completed) and errs == 0 and int(row.manual_submissions) == 1 and row.success_submission_type == "manual":
            p_band = "Low"
        elif bool(row.completed) and 1 <= errs <= 2:
            p_band = "Medium"
        elif errs >= 3:
            p_band = "High"
        else:
            p_band = ""
        if p_band == t_band and (bool(row.completed) or t_band == "High"):
            labels_out[row.episode_id] = t_band
    return labels_out


# Verifikasi konsistensi rekomputasi terhadap episode_labels.csv resmi pada full Train
full_train_people = set(splits_df.loc[splits_df["split"] == "Train", "participant_id"])
recomputed_full = relabel_episodes_inner(episodes_df, full_train_people)
official_labeled = episodes_df.loc[
    episodes_df["cognitive_load_label"].isin(LABELS), ["episode_id", "cognitive_load_label"]
].set_index("episode_id")["cognitive_load_label"].to_dict()
assert recomputed_full == official_labeled, "Rekomputasi behavioral_concordance_v2 tidak cocok 100% dengan CSV resmi!"
print(f"Verifikasi rekomputasi label behavioral_concordance_v2: COCOK 100% ({len(recomputed_full)} episode berlabel).")


def compute_train_weights(sub_df: pd.DataFrame, y_col: str, mode: str) -> np.ndarray:
    """Hitung bobot Train sesuai perlakuan: v3_full_weights, v3_clipped_weights, atau class_balanced."""
    n = len(sub_df)
    w = np.ones(n, dtype=float)
    if mode == "class_balanced":
        vc = sub_df[y_col].value_counts()
        k = len(vc)
        for cls, cnt in vc.items():
            w[sub_df[y_col].to_numpy() == cls] = n / (k * cnt)
        return w

    # Hierarkis: kelas -> peserta -> episode -> window
    classes = [c for c in LABELS if (sub_df[y_col] == c).any()]
    k = len(classes)
    for cls in classes:
        mask_c = (sub_df[y_col] == cls).to_numpy()
        g_c = sub_df.loc[mask_c]
        n_part = g_c["participant_id"].nunique()
        for pid, g_p in g_c.groupby("participant_id"):
            n_ep = g_p["episode_id"].nunique()
            for eid, g_e in g_p.groupby("episode_id"):
                w_val = n / (k * n_part * n_ep * len(g_e))
                idx = (sub_df["episode_id"].to_numpy() == eid) & mask_c
                w[idx] = w_val

    if mode == "v3_clipped_weights":
        # Pangkas ekstrem pada 5.0 lalu normalisasi ulang agar total bobot tiap kelas tetap N / k
        w = np.minimum(w, 5.0)
        for cls in classes:
            mask_c = (sub_df[y_col] == cls).to_numpy()
            w[mask_c] = w[mask_c] * ((n / k) / w[mask_c].sum())
    return w


# Cek cocok dengan train_sample_weights.csv resmi
train_df_check = df[df["split"] == "Train"].reset_index(drop=True)
w_recomp = compute_train_weights(train_df_check, "cognitive_load_label", "v3_full_weights")
assert np.allclose(w_recomp, train_df_check["sample_weight"].to_numpy(), rtol=1e-9, atol=1e-9)
print("Verifikasi rekomputasi bobot v3_full_weights: COCOK 100% dengan train_sample_weights.csv.")


class SensorPreprocessor:
    """Normalisasi fitur sensor bebas kebocoran label."""
    def __init__(self, cols: list, mode: str = "global_robust"):
        self.cols = cols
        self.mode = mode
        self.g_med_ = None
        self.g_iqr_ = None

    def fit(self, fit_df: pd.DataFrame):
        vals = fit_df[self.cols].to_numpy(dtype=float)
        self.g_med_ = np.median(vals, axis=0)
        q75, q25 = np.percentile(vals, [75, 25], axis=0)
        iqr = q75 - q25
        iqr[iqr <= 1e-9] = 1.0
        self.g_iqr_ = iqr
        return self

    def transform(self, target_df: pd.DataFrame) -> np.ndarray:
        vals = target_df[self.cols].to_numpy(dtype=float)
        if self.mode == "global_robust":
            return np.clip((vals - self.g_med_) / self.g_iqr_, -10.0, 10.0)

        out = np.zeros_like(vals, dtype=float)
        if self.mode == "unsupervised_session_robust":
            for sid, idx in target_df.groupby("session_id").indices.items():
                block = vals[idx]
                if len(block) < 3:
                    out[idx] = (block - self.g_med_) / self.g_iqr_
                else:
                    med = np.median(block, axis=0)
                    q75, q25 = np.percentile(block, [75, 25], axis=0)
                    iqr = np.where((q75 - q25) > 1e-9, q75 - q25, self.g_iqr_)
                    out[idx] = (block - med) / iqr
            return np.clip(out, -10.0, 10.0)

        if self.mode == "causal_session_robust":
            # Expanding robust stats hanya dari window sebelumnya pada sesi yang sama (100% kausal)
            for sid, idx in target_df.groupby("session_id").indices.items():
                # Asumsi target_df sudah terurut kronologis per session_id
                idx_sorted = idx[np.argsort(target_df.iloc[idx]["window_start_dt"].to_numpy())]
                for pos, row_i in enumerate(idx_sorted):
                    if pos < 3:
                        out[row_i] = (vals[row_i] - self.g_med_) / self.g_iqr_
                    else:
                        hist = vals[idx_sorted[:pos]]
                        med = np.median(hist, axis=0)
                        q75, q25 = np.percentile(hist, [75, 25], axis=0)
                        iqr = np.where((q75 - q25) > 1e-9, q75 - q25, self.g_iqr_)
                        out[row_i] = (vals[row_i] - med) / iqr
            return np.clip(out, -10.0, 10.0)

        raise ValueError(f"Mode normalisasi tidak dikenal: {self.mode}")


Verifikasi rekomputasi label behavioral_concordance_v2: COCOK 100% (82 episode berlabel).
Verifikasi rekomputasi bobot v3_full_weights: COCOK 100% dengan train_sample_weights.csv.


In [4]:
# Cell 4: Definisi Model (Tabular + PyTorch MLP 3-Seed) & Evaluasi Window + Episode
class SmallTabularMLP(nn.Module):
    def __init__(self, in_dim: int, n_classes: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.LayerNorm(32),
            nn.SiLU(),
            nn.Dropout(0.30),
            nn.Linear(32, 16),
            nn.LayerNorm(16),
            nn.SiLU(),
            nn.Dropout(0.20),
            nn.Linear(16, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def fit_predict_mlp_ensemble(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    w_tr: np.ndarray,
    groups_tr: np.ndarray,
    X_eval: np.ndarray,
    seeds=(42, 43, 44),
    max_epochs: int = 60,
):
    """PyTorch MLP ensemble 3 seed dengan early stopping pada monitor peserta-terpisah di dalam fitting pool."""
    uniq_g = np.unique(groups_tr)
    # Pilih 2 peserta monitor deterministik dari fitting pool (tanpa membaca validation/Test)
    rng_m = np.random.RandomState(SEED)
    perm_g = rng_m.permutation(uniq_g)
    n_mon = max(1, min(2, len(uniq_g) // 5))
    mon_set = set(perm_g[:n_mon])
    mon_mask = np.isin(groups_tr, list(mon_set))
    if mon_mask.sum() < 5 or (~mon_mask).sum() < 10:
        mon_mask = np.zeros(len(y_tr), dtype=bool)

    probs_sum = np.zeros((len(X_eval), len(LABELS)), dtype=float)
    for s in seeds:
        set_all_seeds(s)
        model = SmallTabularMLP(X_tr.shape[1], len(LABELS)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
        loss_fn = nn.CrossEntropyLoss(reduction="none")

        fit_idx = np.where(~mon_mask)[0]
        X_f = torch.tensor(X_tr[fit_idx], dtype=torch.float32, device=DEVICE)
        y_f = torch.tensor(y_tr[fit_idx], dtype=torch.long, device=DEVICE)
        w_f = torch.tensor(w_tr[fit_idx], dtype=torch.float32, device=DEVICE)
        n_fit = len(fit_idx)
        batch_size = 128

        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_loss = float("inf")
        patience, bad = 12, 0

        if mon_mask.any():
            X_m = torch.tensor(X_tr[mon_mask], dtype=torch.float32, device=DEVICE)
            y_m = torch.tensor(y_tr[mon_mask], dtype=torch.long, device=DEVICE)

        for epoch in range(max_epochs):
            model.train()
            perm = torch.randperm(n_fit, device=DEVICE)
            for start in range(0, n_fit, batch_size):
                bidx = perm[start : start + batch_size]
                xb, yb, wb = X_f[bidx], y_f[bidx], w_f[bidx]
                opt.zero_grad()
                logits = model(xb)
                loss = (loss_fn(logits, yb) * wb).sum() / (wb.sum() + 1e-8)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                opt.step()
            sched.step()

            if mon_mask.any():
                model.eval()
                with torch.no_grad():
                    m_loss = float(nn.CrossEntropyLoss()(model(X_m), y_m).item())
                if m_loss < best_loss - 1e-4:
                    best_loss = m_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    bad = 0
                else:
                    bad += 1
                    if bad >= patience:
                        break
            else:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(X_eval, dtype=torch.float32, device=DEVICE))
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        probs_sum += probs
    return probs_sum / len(seeds)


def fit_predict_model(model_name: str, X_tr, y_tr, w_tr, groups_tr, X_eval):
    if model_name == "Dummy":
        clf = DummyClassifier(strategy="most_frequent")
        clf.fit(X_tr, y_tr)
        probs = clf.predict_proba(X_eval)
    elif model_name == "LogisticRegression":
        clf = LogisticRegression(C=0.5, max_iter=5000, random_state=SEED)
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        probs = clf.predict_proba(X_eval)
    elif model_name == "ExtraTrees":
        clf = ExtraTreesClassifier(
            n_estimators=200, min_samples_leaf=5, max_features=0.8, random_state=SEED, n_jobs=1
        )
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        probs = clf.predict_proba(X_eval)
    elif model_name == "HistGradientBoosting":
        clf = HistGradientBoostingClassifier(
            max_iter=120,
            learning_rate=0.05,
            max_leaf_nodes=15,
            min_samples_leaf=10,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=SEED,
        )
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        probs = clf.predict_proba(X_eval)
    elif model_name == "PyTorchMLP":
        return fit_predict_mlp_ensemble(X_tr, y_tr, w_tr, groups_tr, X_eval)
    else:
        raise ValueError(model_name)

    # Pastikan urutan kolom probabilitas tepat [0, 1, 2]
    full_probs = np.zeros((len(X_eval), len(LABELS)), dtype=float)
    for col_idx, cls_val in enumerate(clf.classes_):
        full_probs[:, int(cls_val)] = probs[:, col_idx]
    return full_probs


def evaluate_predictions(y_true_idx: np.ndarray, probs: np.ndarray, episode_ids: np.ndarray) -> dict:
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    probs = probs / probs.sum(axis=1, keepdims=True)
    y_pred = probs.argmax(axis=1)

    recalls = recall_score(y_true_idx, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    win_acc = float(accuracy_score(y_true_idx, y_pred))
    win_bal = float(balanced_accuracy_score(y_true_idx, y_pred))
    win_mf1 = float(f1_score(y_true_idx, y_pred, labels=[0, 1, 2], average="macro", zero_division=0))
    win_ll = float(log_loss(y_true_idx, probs, labels=[0, 1, 2]))

    # Agregasi episode (rata-rata probabilitas per episode_id)
    ep_df = pd.DataFrame({
        "episode_id": episode_ids,
        "y_true": y_true_idx,
        "p0": probs[:, 0],
        "p1": probs[:, 1],
        "p2": probs[:, 2],
    })
    ep_agg = ep_df.groupby("episode_id", as_index=False).agg(
        y_true=("y_true", "first"), p0=("p0", "mean"), p1=("p1", "mean"), p2=("p2", "mean")
    )
    ep_probs = ep_agg[["p0", "p1", "p2"]].to_numpy()
    ep_pred = ep_probs.argmax(axis=1)
    ep_true = ep_agg["y_true"].to_numpy()
    ep_recalls = recall_score(ep_true, ep_pred, labels=[0, 1, 2], average=None, zero_division=0)

    return {
        "n_windows": int(len(y_true_idx)),
        "n_episodes": int(len(ep_agg)),
        "win_accuracy": win_acc,
        "win_bal_acc": win_bal,
        "win_macro_f1": win_mf1,
        "win_log_loss": win_ll,
        "win_recall_low": float(recalls[0]),
        "win_recall_med": float(recalls[1]),
        "win_recall_high": float(recalls[2]),
        "win_min_recall": float(recalls.min()),
        "ep_accuracy": float(accuracy_score(ep_true, ep_pred)),
        "ep_bal_acc": float(balanced_accuracy_score(ep_true, ep_pred)),
        "ep_macro_f1": float(f1_score(ep_true, ep_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "ep_recall_low": float(ep_recalls[0]),
        "ep_recall_med": float(ep_recalls[1]),
        "ep_recall_high": float(ep_recalls[2]),
        "ep_min_recall": float(ep_recalls.min()),
        "all_classes_recalled": bool((recalls > 0).all()),
        "ep_all_classes_recalled": bool((ep_recalls > 0).all()),
    }


def make_episode_summary_frame(window_df: pd.DataFrame, cols: list, norm_mode: str, prep: SensorPreprocessor) -> pd.DataFrame:
    """Agregasi fitur sensor per episode: median, IQR, dan slope linear terhadap waktu dalam episode."""
    X_norm = prep.transform(window_df)
    tmp = pd.DataFrame(X_norm, columns=cols, index=window_df.index)
    tmp["episode_id"] = window_df["episode_id"].to_numpy()
    tmp["participant_id"] = window_df["participant_id"].to_numpy()
    tmp["t_sec"] = (window_df["window_start_dt"] - window_df.groupby("episode_id")["window_start_dt"].transform("min")).dt.total_seconds()
    tmp["label"] = window_df["cognitive_load_label"].to_numpy()

    rows = []
    for eid, g in tmp.groupby("episode_id", sort=True):
        row = {"episode_id": eid, "participant_id": g["participant_id"].iloc[0], "label": g["label"].iloc[0]}
        t = g["t_sec"].to_numpy(dtype=float)
        for c in cols:
            v = g[c].to_numpy(dtype=float)
            row[f"{c}_med"] = float(np.median(v))
            row[f"{c}_iqr"] = float(np.percentile(v, 75) - np.percentile(v, 25))
            if len(v) >= 3 and np.std(t) > 1e-6:
                slope = float(np.polyfit(t / 60.0, v, 1)[0])
            else:
                slope = 0.0
            row[f"{c}_slope"] = float(np.clip(slope, -10.0, 10.0))
        rows.append(row)
    return pd.DataFrame(rows)


In [5]:
# Cell 5: Eksekusi Eksperimen V3 — Inner-CV Bebas Kebocoran + Validation Leaderboard
train_df = df[df["split"] == "Train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "Test"].reset_index(drop=True)

# Siapkan 5-fold participant-grouped di dalam Train dengan rekomputasi label inner-Train
gkf = GroupKFold(n_splits=5)
inner_folds = []
for fold_i, (tr_idx, va_idx) in enumerate(gkf.split(train_df, train_df["cognitive_load_label"], train_df["participant_id"])):
    fit_people = set(train_df.iloc[tr_idx]["participant_id"].unique())
    va_people = set(train_df.iloc[va_idx]["participant_id"].unique())
    # Hitung ulang label episode HANYA dari fit_people
    inner_ep_labels = relabel_episodes_inner(episodes_df, fit_people)
    f_tr = train_df.iloc[tr_idx].copy()
    f_va = train_df.iloc[va_idx].copy()
    f_tr["inner_label"] = f_tr["episode_id"].map(inner_ep_labels)
    f_va["inner_label"] = f_va["episode_id"].map(inner_ep_labels)
    f_tr = f_tr[f_tr["inner_label"].isin(LABELS)].reset_index(drop=True)
    f_va = f_va[f_va["inner_label"].isin(LABELS)].reset_index(drop=True)
    if f_tr["inner_label"].nunique() == 3 and len(f_va) > 0:
        inner_folds.append((fold_i, f_tr, f_va))

print(f"Inner-CV folds valid setelah rekomputasi label inner-Train: {len(inner_folds)}/5")
for fi, f_tr, f_va in inner_folds:
    print(f"  Fold {fi}: fit={len(f_tr)}w ({f_tr['episode_id'].nunique()}ep, {dict(f_tr['inner_label'].value_counts())}) | va={len(f_va)}w ({f_va['episode_id'].nunique()}ep)")

# Grid eksperimen terkontrol (window-level)
MODELS = ["Dummy", "LogisticRegression", "ExtraTrees", "HistGradientBoosting", "PyTorchMLP"]
NORMS = ["global_robust", "causal_session_robust", "unsupervised_session_robust"]
WEIGHT_MODES = ["v3_full_weights", "v3_clipped_weights", "class_balanced"]

# Matriks eksperimen terarah agar tajam & tidak over-search:
# 1) Kontrol Dummy
# 2) Semua model x 3 normalisasi pada core9 + v3_clipped_weights (dan v3_full_weights + class_balanced pada model tabular)
# 3) Ablasi fitur (core10 vs core9 vs core9_head4) pada LogisticRegression, ExtraTrees, HistGradientBoosting
candidate_specs = [
    ("Dummy", "core10", "global_robust", "v3_full_weights", "window"),
]
for norm in NORMS:
    for model_name in ["LogisticRegression", "ExtraTrees", "HistGradientBoosting", "PyTorchMLP"]:
        for w_mode in (WEIGHT_MODES if model_name != "PyTorchMLP" else ["v3_full_weights", "v3_clipped_weights"]):
            candidate_specs.append((model_name, "core9", norm, w_mode, "window"))
for feat_name in ["core10", "core9_head4"]:
    for norm in ["global_robust", "causal_session_robust", "unsupervised_session_robust"]:
        for model_name in ["LogisticRegression", "ExtraTrees", "HistGradientBoosting"]:
            candidate_specs.append((model_name, feat_name, norm, "v3_clipped_weights", "window"))
# Cabang ringkasan episode (episode-level direct classifier)
for feat_name in ["core9", "core9_head4"]:
    for norm in ["global_robust", "unsupervised_session_robust"]:
        for model_name in ["LogisticRegression", "ExtraTrees"]:
            candidate_specs.append((model_name, feat_name, norm, "class_balanced", "episode_summary"))

print(f"Total kandidat eksperimen V3: {len(candidate_specs)}")

cv_rows = []
val_rows = []
t0 = time.time()

# Cache transformasi (feat, norm) per fold & full train/val agar tidak menghitung ulang causal_session_robust
fold_cache = {}
full_cache = {}

def get_fold_xy(fi, f_tr, f_va, feat_name, norm_mode, gran):
    key = (fi, feat_name, norm_mode, gran)
    if key not in fold_cache:
        cols = FEATURE_PROFILES[feat_name]
        prep = SensorPreprocessor(cols, norm_mode).fit(f_tr)
        if gran == "window":
            fold_cache[key] = (prep.transform(f_tr), prep.transform(f_va))
        else:
            tr_ep = make_episode_summary_frame(f_tr.assign(cognitive_load_label=f_tr["inner_label"]), cols, norm_mode, prep)
            va_ep = make_episode_summary_frame(f_va.assign(cognitive_load_label=f_va["inner_label"]), cols, norm_mode, prep)
            fold_cache[key] = (tr_ep, va_ep)
    return fold_cache[key]

def get_full_xy(feat_name, norm_mode, gran):
    key = (feat_name, norm_mode, gran)
    if key not in full_cache:
        cols = FEATURE_PROFILES[feat_name]
        prep = SensorPreprocessor(cols, norm_mode).fit(train_df)
        if gran == "window":
            full_cache[key] = (prep.transform(train_df), prep.transform(val_df))
        else:
            full_cache[key] = (make_episode_summary_frame(train_df, cols, norm_mode, prep), make_episode_summary_frame(val_df, cols, norm_mode, prep))
    return full_cache[key]

for idx_c, (model_name, feat_name, norm_mode, w_mode, gran) in enumerate(candidate_specs, 1):
    cand_id = f"{gran}|{model_name}|{feat_name}|{norm_mode}|{w_mode}"

    # --- A. Leak-free Inner-CV OOF ---
    oof_y, oof_p, oof_ep = [], [], []
    for fi, f_tr, f_va in inner_folds:
        if gran == "window":
            X_tr, X_va = get_fold_xy(fi, f_tr, f_va, feat_name, norm_mode, gran)
            y_tr = f_tr["inner_label"].map(LABEL_TO_IDX).to_numpy()
            y_va = f_va["inner_label"].map(LABEL_TO_IDX).to_numpy()
            w_tr = compute_train_weights(f_tr, "inner_label", w_mode)
            probs = fit_predict_model(model_name, X_tr, y_tr, w_tr, f_tr["participant_id"].to_numpy(), X_va)
            oof_y.append(y_va)
            oof_p.append(probs)
            oof_ep.append(f_va["episode_id"].to_numpy())
        else:
            tr_ep, va_ep = get_fold_xy(fi, f_tr, f_va, feat_name, norm_mode, gran)
            s_cols = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "label")]
            X_tr = tr_ep[s_cols].to_numpy(dtype=float)
            X_va = va_ep[s_cols].to_numpy(dtype=float)
            y_tr = tr_ep["label"].map(LABEL_TO_IDX).to_numpy()
            y_va = va_ep["label"].map(LABEL_TO_IDX).to_numpy()
            vc = pd.Series(y_tr).value_counts()
            w_ep = np.array([len(y_tr) / (len(vc) * vc[y]) for y in y_tr], dtype=float)
            probs = fit_predict_model(model_name, X_tr, y_tr, w_ep, tr_ep["participant_id"].to_numpy(), X_va)
            oof_y.append(y_va)
            oof_p.append(probs)
            oof_ep.append(va_ep["episode_id"].to_numpy())

    cv_metrics = evaluate_predictions(np.concatenate(oof_y), np.vstack(oof_p), np.concatenate(oof_ep))
    cv_rows.append({
        "candidate_id": cand_id, "granularity": gran, "model": model_name,
        "features": feat_name, "normalization": norm_mode, "weighting": w_mode,
        **cv_metrics,
    })

    # --- B. Fit pada full Train -> Evaluasi pada frozen Validation ---
    if gran == "window":
        X_tr, X_va = get_full_xy(feat_name, norm_mode, gran)
        y_tr = train_df["cognitive_load_label"].map(LABEL_TO_IDX).to_numpy()
        y_va = val_df["cognitive_load_label"].map(LABEL_TO_IDX).to_numpy()
        w_tr = compute_train_weights(train_df, "cognitive_load_label", w_mode)
        val_probs = fit_predict_model(model_name, X_tr, y_tr, w_tr, train_df["participant_id"].to_numpy(), X_va)
        val_metrics = evaluate_predictions(y_va, val_probs, val_df["episode_id"].to_numpy())
    else:
        tr_ep, va_ep = get_full_xy(feat_name, norm_mode, gran)
        s_cols = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "label")]
        y_tr = tr_ep["label"].map(LABEL_TO_IDX).to_numpy()
        y_va = va_ep["label"].map(LABEL_TO_IDX).to_numpy()
        vc = pd.Series(y_tr).value_counts()
        w_ep = np.array([len(y_tr) / (len(vc) * vc[y]) for y in y_tr], dtype=float)
        val_probs = fit_predict_model(model_name, tr_ep[s_cols].to_numpy(float), y_tr, w_ep, tr_ep["participant_id"].to_numpy(), va_ep[s_cols].to_numpy(float))
        val_metrics = evaluate_predictions(y_va, val_probs, va_ep["episode_id"].to_numpy())

    val_rows.append({
        "candidate_id": cand_id, "granularity": gran, "model": model_name,
        "features": feat_name, "normalization": norm_mode, "weighting": w_mode,
        **val_metrics,
    })

cv_lb = pd.DataFrame(cv_rows).sort_values(["ep_macro_f1", "win_macro_f1", "win_bal_acc"], ascending=False).reset_index(drop=True)
val_lb = pd.DataFrame(val_rows).sort_values(
    ["ep_all_classes_recalled", "ep_macro_f1", "win_macro_f1", "win_bal_acc"], ascending=False
).reset_index(drop=True)

cv_lb.to_csv(OUTPUT_DIR / "cv_leaderboard.csv", index=False)
val_lb.to_csv(OUTPUT_DIR / "validation_leaderboard.csv", index=False)

print(f"Selesai {len(candidate_specs)} kandidat dalam {time.time()-t0:.1f} detik.")
print("\nTop 10 Validation Leaderboard (V3):")
print(val_lb[["candidate_id", "win_accuracy", "win_macro_f1", "ep_accuracy", "ep_macro_f1", "ep_all_classes_recalled"]].head(10).to_string(index=False))
print("\nTop 10 Leak-Free Inner-CV OOF Leaderboard (V3):")
print(cv_lb[["candidate_id", "win_accuracy", "win_macro_f1", "ep_accuracy", "ep_macro_f1"]].head(10).to_string(index=False))


Inner-CV folds valid setelah rekomputasi label inner-Train: 4/5
  Fold 0: fit=507w (28ep, {'High': np.int64(397), 'Medium': np.int64(58), 'Low': np.int64(52)}) | va=145w (7ep)
  Fold 2: fit=62w (7ep, {'Medium': np.int64(30), 'High': np.int64(26), 'Low': np.int64(6)}) | va=65w (6ep)
  Fold 3: fit=566w (28ep, {'High': np.int64(406), 'Medium': np.int64(114), 'Low': np.int64(46)}) | va=153w (6ep)
  Fold 4: fit=538w (26ep, {'High': np.int64(398), 'Medium': np.int64(99), 'Low': np.int64(41)}) | va=161w (6ep)
Total kandidat eksperimen V3: 60
Selesai 60 kandidat dalam 97.7 detik.

Top 10 Validation Leaderboard (V3):
                                                                             candidate_id  win_accuracy  win_macro_f1  ep_accuracy  ep_macro_f1  ep_all_classes_recalled
episode_summary|LogisticRegression|core9_head4|unsupervised_session_robust|class_balanced      0.818182      0.804233     0.818182     0.804233                     True
              episode_summary|LogisticRegressi

In [6]:
# Cell 6: Penguncian Pemenang Sebelum Test & Satu Evaluasi Akhir pada Frozen Test
# Gabungkan skor CV & Validation untuk pemeriksaan stabilitas, pilih pemenang dari Validation (bukan Dummy)
# PENTING: Batasi seleksi pemenang utama HANYA ke model per-window (`granularity == "window"`)
# karena cabang `episode_summary` (iqr/slope) bernilai tepat 0 pada episode pendek (1-2 window)
# sehingga secara implisit membocorkan durasi episode ke label V3 yang berbasis durasi.
eligible = val_lb[(val_lb["model"] != "Dummy") & (val_lb["granularity"] == "window")].copy()
deployable = eligible[eligible["normalization"].isin(["global_robust", "causal_session_robust"])].copy()
merged_sel = deployable.merge(
    cv_lb[["candidate_id", "ep_macro_f1", "win_macro_f1"]].rename(
        columns={"ep_macro_f1": "oof_ep_macro_f1", "win_macro_f1": "oof_win_macro_f1"}
    ),
    on="candidate_id",
    how="left",
)
# Peringkat: recall semua kelas di validation -> rata-rata stabil (val ep_macro_f1 & win_macro_f1) -> OOF ep_macro_f1
merged_sel["selection_score"] = 0.5 * merged_sel["ep_macro_f1"] + 0.3 * merged_sel["win_macro_f1"] + 0.2 * merged_sel["oof_ep_macro_f1"]
merged_sel = merged_sel.sort_values(
    ["ep_all_classes_recalled", "selection_score", "ep_macro_f1", "win_macro_f1"], ascending=False
).reset_index(drop=True)

winner = merged_sel.iloc[0]
print("PEMENANG TERKUNCI (sebelum membuka Test):")
print(f"  candidate_id  : {winner['candidate_id']}")
print(f"  Val win_mF1   : {winner['win_macro_f1']:.4f} | Val ep_mF1 : {winner['ep_macro_f1']:.4f}")
print(f"  OOF win_mF1   : {winner['oof_win_macro_f1']:.4f} | OOF ep_mF1 : {winner['oof_ep_macro_f1']:.4f}")

def eval_on_test(row_spec):
    cols = FEATURE_PROFILES[row_spec["features"]]
    prep = SensorPreprocessor(cols, row_spec["normalization"]).fit(train_df)
    if row_spec["granularity"] == "window":
        X_tr = prep.transform(train_df)
        X_te = prep.transform(test_df)
        y_tr = train_df["cognitive_load_label"].map(LABEL_TO_IDX).to_numpy()
        y_te = test_df["cognitive_load_label"].map(LABEL_TO_IDX).to_numpy()
        w_tr = compute_train_weights(train_df, "cognitive_load_label", row_spec["weighting"])
        probs = fit_predict_model(row_spec["model"], X_tr, y_tr, w_tr, train_df["participant_id"].to_numpy(), X_te)
        metrics = evaluate_predictions(y_te, probs, test_df["episode_id"].to_numpy())
        pred_df = test_df[["window_id", "participant_id", "session_id", "episode_id", "cognitive_load_label"]].copy()
        pred_df["pred_label"] = [LABELS[i] for i in probs.argmax(axis=1)]
        for i, lbl in enumerate(LABELS):
            pred_df[f"prob_{lbl}"] = probs[:, i]
        return metrics, pred_df, y_te, probs
    else:
        tr_ep = make_episode_summary_frame(train_df, cols, row_spec["normalization"], prep)
        te_ep = make_episode_summary_frame(test_df, cols, row_spec["normalization"], prep)
        s_cols = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "label")]
        y_tr = tr_ep["label"].map(LABEL_TO_IDX).to_numpy()
        y_te = te_ep["label"].map(LABEL_TO_IDX).to_numpy()
        vc = pd.Series(y_tr).value_counts()
        w_ep = np.array([len(y_tr) / (len(vc) * vc[y]) for y in y_tr], dtype=float)
        probs = fit_predict_model(row_spec["model"], tr_ep[s_cols].to_numpy(float), y_tr, w_ep, tr_ep["participant_id"].to_numpy(), te_ep[s_cols].to_numpy(float))
        metrics = evaluate_predictions(y_te, probs, te_ep["episode_id"].to_numpy())
        pred_df = te_ep[["episode_id", "participant_id"]].copy()
        pred_df["cognitive_load_label"] = te_ep["label"]
        pred_df["pred_label"] = [LABELS[i] for i in probs.argmax(axis=1)]
        for i, lbl in enumerate(LABELS):
            pred_df[f"prob_{lbl}"] = probs[:, i]
        return metrics, pred_df, y_te, probs

test_winner_metrics, test_pred_df, y_te_idx, probs_te = eval_on_test(winner)
dummy_spec = val_lb[val_lb["model"] == "Dummy"].iloc[0]
test_dummy_metrics, _, _, _ = eval_on_test(dummy_spec)

# Catat juga kandidat terbaik unsupervised_session_robust hanya sebagai referensi kalibrasi penuh
unsup_best = eligible[eligible["normalization"] == "unsupervised_session_robust"].iloc[0]
test_unsup_metrics, _, _, _ = eval_on_test(unsup_best)

print("\n=== HASIL FINAL TEST V3 (MODEL TERKUNCI — DIEVALUASI SEKALI) ===")
print(f"Locked Winner ({winner['candidate_id']}):")
print(f"  Window  -> Acc={test_winner_metrics['win_accuracy']:.2%}, BalAcc={test_winner_metrics['win_bal_acc']:.2%}, Macro-F1={test_winner_metrics['win_macro_f1']:.2%}, Recalls(L/M/H)=({test_winner_metrics['win_recall_low']:.2%}/{test_winner_metrics['win_recall_med']:.2%}/{test_winner_metrics['win_recall_high']:.2%})")
print(f"  Episode -> Acc={test_winner_metrics['ep_accuracy']:.2%}, BalAcc={test_winner_metrics['ep_bal_acc']:.2%}, Macro-F1={test_winner_metrics['ep_macro_f1']:.2%}, Recalls(L/M/H)=({test_winner_metrics['ep_recall_low']:.2%}/{test_winner_metrics['ep_recall_med']:.2%}/{test_winner_metrics['ep_recall_high']:.2%})")
print(f"Dummy Control -> Window Acc={test_dummy_metrics['win_accuracy']:.2%}, Macro-F1={test_dummy_metrics['win_macro_f1']:.2%} | Episode Acc={test_dummy_metrics['ep_accuracy']:.2%}, Macro-F1={test_dummy_metrics['ep_macro_f1']:.2%}")
print(f"Unsup-Session Reference ({unsup_best['candidate_id']}) -> Window Macro-F1={test_unsup_metrics['win_macro_f1']:.2%} | Episode Macro-F1={test_unsup_metrics['ep_macro_f1']:.2%}")

test_pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

# Diagnostik per episode pada Test
if "window_id" in test_pred_df.columns:
    ep_diag = test_pred_df.groupby(["episode_id", "participant_id", "cognitive_load_label"], as_index=False).agg(
        n_windows=("window_id", "count"),
        prob_Low=("prob_Low", "mean"),
        prob_Medium=("prob_Medium", "mean"),
        prob_High=("prob_High", "mean"),
    )
    ep_diag["ep_pred"] = [LABELS[i] for i in ep_diag[["prob_Low", "prob_Medium", "prob_High"]].to_numpy().argmax(axis=1)]
else:
    ep_diag = test_pred_df.rename(columns={"pred_label": "ep_pred"}).copy()
    ep_diag["n_windows"] = ep_diag["episode_id"].map(test_df.groupby("episode_id").size())
ep_diag["correct"] = ep_diag["cognitive_load_label"] == ep_diag["ep_pred"]
ep_diag.to_csv(OUTPUT_DIR / "episode_diagnostics.csv", index=False)

test_eval_payload = {
    "locked_winner": {
        "candidate_id": winner["candidate_id"],
        "val_metrics": {k: (float(winner[k]) if isinstance(winner[k], (np.floating, float)) else int(winner[k]) if isinstance(winner[k], (np.integer, int)) else bool(winner[k]) if isinstance(winner[k], (np.bool_, bool)) else winner[k]) for k in winner.index},
        "test_metrics": test_winner_metrics,
    },
    "dummy_control": {
        "candidate_id": dummy_spec["candidate_id"],
        "test_metrics": test_dummy_metrics,
    },
    "unsupervised_session_reference": {
        "candidate_id": unsup_best["candidate_id"],
        "test_metrics": test_unsup_metrics,
    },
}
(OUTPUT_DIR / "test_evaluation.json").write_text(json.dumps(test_eval_payload, indent=2), encoding="utf-8")

run_meta = {
    "experiment": "batch-06/exp01_v3_behavioral_proxy",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "device": DEVICE,
    "seed": SEED,
    "dataset_sha256": file_hashes,
    "n_candidates": int(len(candidate_specs)),
    "locked_winner_id": winner["candidate_id"],
}
(OUTPUT_DIR / "run.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")


PEMENANG TERKUNCI (sebelum membuka Test):
  candidate_id  : window|LogisticRegression|core9|causal_session_robust|class_balanced
  Val win_mF1   : 0.4212 | Val ep_mF1 : 0.6429
  OOF win_mF1   : 0.2967 | OOF ep_mF1 : 0.4718

=== HASIL FINAL TEST V3 (MODEL TERKUNCI — DIEVALUASI SEKALI) ===
Locked Winner (window|LogisticRegression|core9|causal_session_robust|class_balanced):
  Window  -> Acc=35.50%, BalAcc=34.05%, Macro-F1=33.50%, Recalls(L/M/H)=(30.65%/32.00%/39.50%)
  Episode -> Acc=33.33%, BalAcc=36.11%, Macro-F1=30.13%, Recalls(L/M/H)=(33.33%/25.00%/50.00%)
Dummy Control -> Window Acc=51.52%, Macro-F1=22.67% | Episode Acc=11.11%, Macro-F1=6.67%
Unsup-Session Reference (window|LogisticRegression|core9|unsupervised_session_robust|v3_clipped_weights) -> Window Macro-F1=21.07% | Episode Macro-F1=11.76%


In [7]:
# Cell 7: Visualisasi & Laporan RESULTS.md
# 1. Perbandingan Ablasi pada Validation & OOF
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_val = val_lb[val_lb["model"] != "Dummy"].head(12).iloc[::-1]
axes[0].barh(range(len(top_val)), top_val["ep_macro_f1"], alpha=0.85, label="Episode Macro-F1", color="#2563eb")
axes[0].barh(range(len(top_val)), top_val["win_macro_f1"], alpha=0.55, label="Window Macro-F1", color="#10b981")
axes[0].set_yticks(range(len(top_val)))
axes[0].set_yticklabels([c.replace("window|", "").replace("episode_summary|", "ep|") for c in top_val["candidate_id"]], fontsize=8)
axes[0].axvline(1/3, color="red", linestyle="--", linewidth=1, label="Random (33.3%)")
axes[0].set_xlabel("Macro-F1")
axes[0].set_title("V3 Validation: Top 12 Kandidat")
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 1.0)

# Ablasi bobot pada Validation (rata-rata Macro-F1 window & episode per weighting mode)
w_agg = val_lb[(val_lb["granularity"] == "window") & (val_lb["model"] != "Dummy") & (val_lb["features"] == "core9")].groupby("weighting")[["win_macro_f1", "ep_macro_f1"]].mean()
x_pos = np.arange(len(w_agg))
axes[1].bar(x_pos - 0.18, w_agg["win_macro_f1"], width=0.35, label="Mean Val Window Macro-F1", color="#10b981")
axes[1].bar(x_pos + 0.18, w_agg["ep_macro_f1"], width=0.35, label="Mean Val Episode Macro-F1", color="#2563eb")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(w_agg.index, rotation=15, fontsize=9)
axes[1].axhline(1/3, color="red", linestyle="--", linewidth=1, label="Random (33.3%)")
axes[1].set_ylabel("Macro-F1")
axes[1].set_title("V3 Ablasi Pembobotan Train (Core9)")
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 0.7)
plt.tight_layout()
plt.savefig(FIG_DIR / "v3_leaderboard_comparison.png", dpi=150)
plt.close(fig)

# 2. Confusion Matrix Test (Window & Episode)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
cm_win = confusion_matrix(
    test_pred_df["cognitive_load_label"].map(LABEL_TO_IDX),
    test_pred_df["pred_label"].map(LABEL_TO_IDX),
    labels=[0, 1, 2],
)
cm_ep = confusion_matrix(
    ep_diag["cognitive_load_label"].map(LABEL_TO_IDX),
    ep_diag["ep_pred"].map(LABEL_TO_IDX),
    labels=[0, 1, 2],
)
for ax, cm, title in [(axes[0], cm_win, f"Test Window CM (n={cm_win.sum()})"), (axes[1], cm_ep, f"Test Episode CM (n={cm_ep.sum()})")]:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(LABELS)
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(LABELS)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=11)
plt.suptitle(f"V3 Locked Winner: {winner['candidate_id']}", fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "v3_confusion_matrices.png", dpi=150)
plt.close(fig)

# 3. Tulis RESULTS.md
md_lines = [
    "# Hasil Eksperimen Batch 06 — Exp01: `v3_behavioral_proxy`",
    "",
    f"- **Tanggal run (UTC):** {run_meta['generated_at_utc']}",
    f"- **Jumlah kandidat diuji:** {len(candidate_specs)}",
    f"- **Model terkunci (dipilih sebelum Test):** `{winner['candidate_id']}`",
    "",
    "## Ringkasan Performa Model Terkunci",
    "",
    "| Split | Window Acc | Window BalAcc | Window Macro-F1 | Episode Acc | Episode BalAcc | Episode Macro-F1 |",
    "|---|---:|---:|---:|---:|---:|---:|",
    f"| Inner-CV OOF (Train, leak-free) | — | — | {winner['oof_win_macro_f1']:.2%} | — | — | {winner['oof_ep_macro_f1']:.2%} |",
    f"| Frozen Validation (3 org, 11 ep) | {winner['win_accuracy']:.2%} | {winner['win_bal_acc']:.2%} | {winner['win_macro_f1']:.2%} | {winner['ep_accuracy']:.2%} | {winner['ep_bal_acc']:.2%} | {winner['ep_macro_f1']:.2%} |",
    f"| **Frozen Test (3 org, 18 ep)** | **{test_winner_metrics['win_accuracy']:.2%}** | **{test_winner_metrics['win_bal_acc']:.2%}** | **{test_winner_metrics['win_macro_f1']:.2%}** | **{test_winner_metrics['ep_accuracy']:.2%}** | **{test_winner_metrics['ep_bal_acc']:.2%}** | **{test_winner_metrics['ep_macro_f1']:.2%}** |",
    f"| Dummy Control (Test) | {test_dummy_metrics['win_accuracy']:.2%} | {test_dummy_metrics['win_bal_acc']:.2%} | {test_dummy_metrics['win_macro_f1']:.2%} | {test_dummy_metrics['ep_accuracy']:.2%} | {test_dummy_metrics['ep_bal_acc']:.2%} | {test_dummy_metrics['ep_macro_f1']:.2%} |",
    f"| Unsupervised Session-Norm Ref (Test) | {test_unsup_metrics['win_accuracy']:.2%} | {test_unsup_metrics['win_bal_acc']:.2%} | {test_unsup_metrics['win_macro_f1']:.2%} | {test_unsup_metrics['ep_accuracy']:.2%} | {test_unsup_metrics['ep_bal_acc']:.2%} | {test_unsup_metrics['ep_macro_f1']:.2%} |",
    "",
    "## Recall per Kelas pada Test (Model Terkunci)",
    "",
    f"- **Window Recall (Low / Medium / High):** {test_winner_metrics['win_recall_low']:.2%} / {test_winner_metrics['win_recall_med']:.2%} / {test_winner_metrics['win_recall_high']:.2%}",
    f"- **Episode Recall (Low / Medium / High):** {test_winner_metrics['ep_recall_low']:.2%} / {test_winner_metrics['ep_recall_med']:.2%} / {test_winner_metrics['ep_recall_high']:.2%}",
    "",
    "## Catatan Evaluasi",
    "",
    "- Kelas High pada Test V3 hanya berasal dari **2 episode** (119 window overlap), sehingga variasi episode sangat besar.",
    "- Cabang `episode_summary` (yang menghitung IQR/slope lintas window dalam satu episode) disimpan hanya sebagai diagnostik dan **dikeluarkan dari seleksi pemenang utama** karena episode pendek 1–2 window menghasilkan nilai IQR/slope tepat 0, sehingga secara implisit membocorkan durasi episode ke label V3 yang berbasis durasi.",
    "- Semua angka di atas dihitung dari model murni **per-window** tanpa kebocoran peserta, tanpa kebocoran referensi waktu pada inner-CV, dan tanpa mengganti pemenang setelah membuka Test.",
    "",
    "![Leaderboard V3](figures/v3_leaderboard_comparison.png)",
    "",
    "![Confusion Matrix V3](figures/v3_confusion_matrices.png)",
]
(OUTPUT_DIR / "RESULTS.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print("Artefak V3 lengkap tersimpan di:", OUTPUT_DIR)


Artefak V3 lengkap tersimpan di: E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp01_v3_behavioral_proxy
